In [ ]:
import hydra
from omegaconf import OmegaConf
from aereo.executors import LocalExecutor
from aereo.pipeline import ExtractionJob

# Load the job model (grid, patch, extract stages) from the Hydra config package.
# Search provider and task builder are runtime plugins and are loaded separately.
job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_sentinel2",
)

search_cfg = OmegaConf.load("config/search/sentinel2_pc.yaml")
search_provider = hydra.utils.instantiate(search_cfg, _convert_="all", _partial_=True)

task_builder_cfg = OmegaConf.load("config/task_builder/grouped.yaml")
task_builder = hydra.utils.instantiate(
    task_builder_cfg, _convert_="all", _partial_=True
)

In [ ]:
# Search
search_results = job.search(search_provider)
print(f"✓ Found {len(search_results)} scenes")

In [ ]:
# Build tasks
print("\n📦 Building tasks...")
tasks = job.build_tasks(search_results, task_builder)
print(f"✓ Built {len(tasks)} tasks")

In [ ]:
# Extract
print("\n⛏️ Extracting...")
artifacts = job.execute(tasks, executor=LocalExecutor(workers=8))
print(f"✓ Extracted {len(artifacts)} artifacts")

In [ ]:
from aereo.viz import plot_artifact_patches

In [ ]:
plot_artifact_patches(artifacts, ds_factor=10, cmap="viridis")